A Tool is a function given to the LLM. This function should fulfill a clear objective.

Here are some commonly used tools in AI agents:

Tool	                    Description

Web Search	                Allows the agent to fetch up-to-date information from the internet.

Image Generation	        Creates images based on text descriptions.

Retrieval	                Retrieves information from an external source.

API Interface	            Interacts with an external API (GitHub, YouTube, Spotify, etc.).

Those are only examples, as you can in fact create a tool for any use case!

A good tool should be something that complements the power of an LLM.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
os.environ['GEMINI_API_KEY']=os.getenv('GEMINI_API_KEY')

In [2]:
import os
from langchain.chat_models import init_chat_model
# model = init_chat_model("groq:qwen/qwen3.6-27b")
model = init_chat_model("groq:openai/gpt-oss-120b")
response=model.invoke("Hellow, how are you?")
print(response)

content='Hello! I’m doing great, thank you for asking. How can I assist you today?' additional_kwargs={'reasoning_content': 'The user says "Hellow, how are you?" It\'s a casual greeting. The assistant should respond politely. No policy issues. Just respond.'} response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 78, 'total_tokens': 136, 'completion_time': 0.120095835, 'completion_tokens_details': {'reasoning_tokens': 30}, 'prompt_time': 0.00354416, 'prompt_tokens_details': None, 'queue_time': 0.709367658, 'total_time': 0.123639995}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bb691ea66b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a06b3d-7305-7651-8f1e-ab71e7737747-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 78, 'output_tokens': 58, 'total_tokens': 136, 'output_token_details': {'reasoning': 30}}


In [3]:
from langchain.tools import tool

@tool
def get_current_weather(location: str) -> str:
    """ Returns the current weather in a given location"""
    return f"The current weather in {location} is nice"

In [4]:
model_with_tools = model.bind_tools([get_current_weather])

In [5]:
response = model_with_tools.invoke("Hellow, whta is the weather in New York?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool : {tool_call['name']}")
    print(f"Args : {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'The user asks: "Hellow, whta is the weather in New York?" Need to fetch current weather using function get_current_weather. Use function call.', 'tool_calls': [{'id': 'fc_edefa871-977d-4f25-a9a1-d46f3df660cb', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_current_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 135, 'total_tokens': 197, 'completion_time': 0.13182843, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.00582423, 'prompt_tokens_details': None, 'queue_time': 0.386793798, 'total_time': 0.13765266}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_803c0ba83d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a06b40-819a-7c51-b47b-7da8979fc579-0' tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'New York'}, 'id': 'fc_edefa871-9

#### Tool execution

In [6]:
#Step 1: model generates tool calls
messages = [{"role": "user", "content": "Hellow, whta is the weather in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2: Execute tools and collect the results
for tool_call in ai_msg.tool_calls:
    tool_result = get_current_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pas results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response)

content='Sure thing! The current weather in New\u202fYork is nice. Let me know if you’d like more details—like temperature, humidity, or a forecast for the next few days.' additional_kwargs={'reasoning_content': 'The user asked "Hellow, whta is the weather in New York?" The assistant called get_current_weather function with location "New York". The function returned "The current weather in New York is nice". That\'s a placeholder. The assistant should respond with that info in a friendly manner, perhaps ask if they\'d like more details.'} response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 171, 'total_tokens': 284, 'completion_time': 0.232260488, 'completion_tokens_details': {'reasoning_tokens': 67}, 'prompt_time': 0.006858654, 'prompt_tokens_details': None, 'queue_time': 0.388351695, 'total_time': 0.239119142}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bb691ea66b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'm